# [KIỆT] Model Implementation & Comparison
# Updated by Kiet

**Mục tiêu:** Huấn luyện và so sánh 5 mô hình Machine Learning trên bộ dữ liệu CIC-IDS2017.

**Đầu vào:** `data/processed/X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`, `models/label_encoder.pkl`  
**Đầu ra:** `models/model_results.csv`, `models/random_forest.pkl`

## [KIỆT] - Bước 1: Import thư viện

In [1]:
import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score
)

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'

print('Thư viện đã được tải thành công.')

Thư viện đã được tải thành công.


## [KIỆT] - Bước 2: Tải dữ liệu

In [2]:
X_train = np.load(PROCESSED_DIR / 'X_train.npy')
X_test  = np.load(PROCESSED_DIR / 'X_test.npy')
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
y_test  = np.load(PROCESSED_DIR / 'y_test.npy')

le = joblib.load(MODELS_DIR / 'label_encoder.pkl')
target_names = le.classes_.tolist()

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')
print(f'Nhãn ({len(target_names)}): {target_names}')

X_train: (3458107, 18), X_test: (566149, 18)
y_train: (3458107,), y_test: (566149,)
Nhãn (15): ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']


## [KIỆT] - Bước 3: Định nghĩa 5 mô hình

In [3]:
from sklearn.linear_model import SGDClassifier

models = [
    ('Logistic Regression', LogisticRegression(
        solver='saga', max_iter=100, tol=1e-2, n_jobs=-1, random_state=42
    )),
    ('SVM', SGDClassifier(
        loss='hinge', max_iter=100, tol=1e-2, n_jobs=-1, random_state=42
    )),
    ('Naive Bayes', GaussianNB()),
    ('KNN', KNeighborsClassifier(
        n_neighbors=5, n_jobs=-1
    )),
    ('Random Forest', RandomForestClassifier(
        n_estimators=50, random_state=42, n_jobs=-1
    )),
]

MODEL_SAMPLE_CAP = {
    'Logistic Regression': 200_000,
    'SVM':                 100_000,
    'KNN':                 100_000,
    'Random Forest':       300_000,
}

print(f'Se huan luyen {len(models)} mo hinh:')
for name, _ in models:
    cap = MODEL_SAMPLE_CAP.get(name)
    note = f' (cap {cap:,} mau)' if cap else ''
    print(f'  - {name}{note}')


Se huan luyen 5 mo hinh:
  - Logistic Regression (cap 200,000 mau)
  - SVM (cap 100,000 mau)
  - Naive Bayes
  - KNN (cap 100,000 mau)
  - Random Forest (cap 300,000 mau)


## [KIỆT] - Bước 4: Huấn luyện và đánh giá từng mô hình

In [4]:
results = []
trained_models = {}

for model_name, model in models:
    print('=' * 60)
    print(f'[KIET] Dang huan luyen: {model_name}')
    print('=' * 60)

    cap = MODEL_SAMPLE_CAP.get(model_name)
    if cap and len(X_train) > cap:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X_train), cap, replace=False)
        X_fit, y_fit = X_train[idx], y_train[idx]
        print(f'  Dung {cap:,} / {len(X_train):,} mau')
    else:
        X_fit, y_fit = X_train, y_train

    t_start = time.time()
    model.fit(X_fit, y_fit)
    t_train = time.time() - t_start

    y_pred = model.predict(X_test)

    print(f'Thoi gian huan luyen: {t_train:.2f}s')
    print('Classification Report:')
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    results.append({
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1 Score':  round(f1,   4),
        'Train Time (s)': round(t_train, 2),
    })

    trained_models[model_name] = (model, y_pred)
    print(f'  Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}')


[KIET] Dang huan luyen: Logistic Regression
  Dung 200,000 / 3,458,107 mau
Thoi gian huan luyen: 13.61s
Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.75      0.85    454620
                       Bot       0.00      0.02      0.01       393
                      DDoS       0.42      0.86      0.57     25606
             DoS GoldenEye       0.59      0.64      0.61      2059
                  DoS Hulk       0.84      0.80      0.82     46215
          DoS Slowhttptest       0.02      0.32      0.04      1100
             DoS slowloris       0.09      0.21      0.13      1159
               FTP-Patator       0.09      0.49      0.15      1588
                Heartbleed       0.02      1.00      0.03         2
              Infiltration       0.00      1.00      0.01         7
                  PortScan       0.70      0.94      0.80     31786
               SSH-Patator       0.04      0.50      0.0

## [KIỆT] - Bước 5: Bảng tổng hợp kết quả

In [5]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print('[KIỆT] Bảng so sánh kết quả 5 mô hình (sắp xếp theo F1):')
print(results_df.to_string(index=False))

[KIỆT] Bảng so sánh kết quả 5 mô hình (sắp xếp theo F1):
              Model  Accuracy  Precision  Recall  F1 Score  Train Time (s)
      Random Forest    0.9873     0.9964  0.9873    0.9914            4.92
                KNN    0.9610     0.9801  0.9610    0.9691            0.01
Logistic Regression    0.7623     0.9258  0.7623    0.8245           13.61
                SVM    0.7470     0.9219  0.7470    0.8102            0.26
        Naive Bayes    0.2793     0.9365  0.2793    0.3844            0.99


## [KIỆT] - Bước 6: Lưu kết quả và mô hình Random Forest

In [6]:
# Lưu bảng kết quả
results_path = MODELS_DIR / 'model_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Đã lưu kết quả: {results_path}')

# Lưu Random Forest model
rf_model, _ = trained_models['Random Forest']
rf_path = MODELS_DIR / 'random_forest.pkl'
joblib.dump(rf_model, rf_path)
print(f'Đã lưu Random Forest: {rf_path}')

# Lưu tất cả models để notebook 04 có thể dùng
all_preds = {name: pred for name, (_, pred) in trained_models.items()}
joblib.dump(all_preds, MODELS_DIR / 'all_predictions.pkl')

all_fitted = {name: model for name, (model, _) in trained_models.items()}
joblib.dump(all_fitted, MODELS_DIR / 'all_models.pkl')
print(f'Đã lưu tất cả predictions và models.')

print('\n✓ [KIỆT] Hoàn thành Model Training!')

Đã lưu kết quả: D:\javabtap\Network-Intrusion-Detection-ML\models\model_results.csv
Đã lưu Random Forest: D:\javabtap\Network-Intrusion-Detection-ML\models\random_forest.pkl
Đã lưu tất cả predictions và models.

✓ [KIỆT] Hoàn thành Model Training!
